# M-dwarf spectrum

In [6]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize

from vizlib.animation_export import export_animation


# =========================
# EXPORT CONFIG
# =========================

OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/dwarf_spectra")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FPS = 30
DURATION_SEC = 10
N_FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 5)
DPI = 140


# =========================
# DATA CONFIG
# =========================

X_MIN, X_MAX = 0.45, 14.0
Y_MIN, Y_MAX = 0.008, 1.25

rng = np.random.default_rng(42)
x = np.linspace(X_MIN, X_MAX, 4200)


# =========================
# SYNTHETIC M-DWARF SPECTRUM
# =========================

continuum = (
    0.10
    + 0.55 * np.exp(-((x - 0.75) / 0.28) ** 2)
    + 0.42 * np.exp(-((x - 1.15) / 0.35) ** 2)
    + 0.30 * np.exp(-((x - 1.75) / 0.45) ** 2)
    + 0.20 * np.exp(-((x - 3.7) / 1.7) ** 2)
    + 0.08 * np.exp(-((x - 7.0) / 3.2) ** 2)
)

continuum *= np.exp(-0.055 * np.clip(x - 1.0, 0, None))

bands = [
    (0.55, 0.025, 0.35),
    (0.74, 0.035, 0.28),
    (0.93, 0.055, 0.55),
    (1.35, 0.075, 0.60),
    (1.90, 0.110, 0.65),
    (2.32, 0.085, 0.50),
    (2.70, 0.160, 0.60),
    (3.30, 0.130, 0.45),
    (4.30, 0.180, 0.55),
    (5.00, 0.250, 0.70),
    (7.60, 0.450, 0.45),
    (9.20, 0.600, 0.55),
    (11.5, 0.900, 0.45),
]

flux = continuum.copy()

for center, width, depth in bands:
    flux *= 1.0 - depth * np.exp(-0.5 * ((x - center) / width) ** 2)

fine = (
    1
    + 0.10 * np.sin(85 * x)
    + 0.055 * np.sin(190 * x + 0.4)
    + 0.035 * np.sin(420 * x + 1.7)
)

noise = rng.normal(0, 0.045, size=x.size)
noise *= np.exp(-0.10 * (x - 0.5))

flux *= fine * (1 + noise)

line_centers = np.concatenate([
    rng.uniform(0.5, 1.4, 90),
    rng.uniform(1.4, 2.8, 70),
    rng.uniform(2.8, 6.0, 55),
    rng.uniform(6.0, 13.5, 45),
])

for c in line_centers:
    width = rng.uniform(0.003, 0.018)
    depth = rng.uniform(0.08, 0.45)
    flux *= 1.0 - depth * np.exp(-0.5 * ((x - c) / width) ** 2)

flux = np.clip(flux, Y_MIN * 1.05, Y_MAX)


# =========================
# FIGURE
# =========================

cmap = LinearSegmentedColormap.from_list(
    "m_dwarf_fire",
    ["#ffb347", "#ff8c1a", "#ff4a1f", "#b00014", "#300006"],
)

norm = Normalize(X_MIN, X_MAX)

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

fig.patch.set_facecolor("#030303")
ax.set_facecolor("#030303")

ax.set_xlim(X_MIN, X_MAX)
ax.set_ylim(Y_MIN, Y_MAX)
ax.set_yscale("log")

ax.set_title(
    "ТИПИЧНЫЙ СПЕКТР M-КАРЛИКА",
    loc="left",
    fontsize=19,
    color="#c79a6c",
    pad=18,
    weight="bold",
)

ax.set_xlabel("Длина волны (мкм)", fontsize=13, color="#b99b82", labelpad=12)
ax.set_ylabel("Относительный поток", fontsize=13, color="#b99b82", labelpad=12)

ax.set_xticks([0.5, 1, 2, 5, 10])
ax.set_xticklabels(["0.5", "1", "2", "5", "10"], color="#c9b39d", fontsize=12)

ax.set_yticks([0.01, 0.1, 1.0])
ax.set_yticklabels(["0.01", "0.1", "1.0"], color="#c9b39d", fontsize=12)

for spine in ax.spines.values():
    spine.set_color("#6b523b")
    spine.set_linewidth(1.0)

ax.tick_params(colors="#c9b39d", which="both", length=6, width=1)
ax.grid(False)

ax.axhline(Y_MIN, color="#3b281d", lw=1.2)
ax.axvline(X_MIN, color="#3b281d", lw=1.2)

fill_poly = ax.fill_between([], [], [], color="#ff4a1f", alpha=0.10)

line_collection = LineCollection([], linewidths=1.4, alpha=0.95)
line_collection.set_cmap(cmap)
line_collection.set_norm(norm)
ax.add_collection(line_collection)

glow_collection = LineCollection([], linewidths=5.0, alpha=0.16)
glow_collection.set_cmap(cmap)
glow_collection.set_norm(norm)
ax.add_collection(glow_collection)


# =========================
# LABELS
# =========================

labels = [
    (0.62, 0.62, "TiO\nVO", 0.58, 0.16),
    (1.03, 0.45, "H$_2$O\nFeH", 0.90, 0.14),
    (1.70, 0.55, "H$_2$O", 1.55, 0.16),
    (2.25, 0.52, "CO", 2.08, 0.17),
    (3.15, 0.40, "H$_2$O", 2.80, 0.12),
    (5.25, 0.38, "CO\nH$_2$O", 4.80, 0.10),
    (9.40, 0.35, "CH$_4$\nH$_2$O", 8.70, 0.08),
]

label_artists = []

for tx, ty, text, px, py in labels:
    ann = ax.annotate(
        text,
        xy=(px, py),
        xytext=(tx, ty),
        color="#c9a57f",
        fontsize=13,
        ha="center",
        va="center",
        arrowprops=dict(
            arrowstyle="-",
            color="#9d7a5b",
            lw=1.0,
            shrinkA=4,
            shrinkB=4,
            alpha=0.0,
        ),
        alpha=0.0,
    )
    label_artists.append((ann, tx))


# =========================
# ANIMATION HELPERS
# =========================

def build_segments(xv, yv):
    points = np.column_stack([xv, yv]).reshape(-1, 1, 2)
    return np.concatenate([points[:-1], points[1:]], axis=1)


def smoothstep(t):
    return t * t * (3 - 2 * t)


def fig_to_array(fig, alpha=False):
    fig.canvas.draw()

    rgba = np.asarray(fig.canvas.buffer_rgba(), dtype=np.uint8)

    if alpha:
        return rgba.copy()

    return rgba[..., :3].copy()


def update_frame(frame_index: int):
    global fill_poly

    t = frame_index / max(N_FRAMES - 1, 1)
    reveal = smoothstep(t)
    x_cut = X_MIN + (X_MAX - X_MIN) * reveal

    mask = x <= x_cut
    xv = x[mask]
    yv = flux[mask]

    if len(xv) > 3:
        segments = build_segments(xv, yv)

        line_collection.set_segments(segments)
        line_collection.set_array(xv[:-1])

        glow_collection.set_segments(segments)
        glow_collection.set_array(xv[:-1])

        fill_poly.remove()
        fill_poly = ax.fill_between(
            xv,
            yv,
            Y_MIN,
            color="#ff4a1f",
            alpha=0.08,
        )

    for ann, label_x in label_artists:
        fade = np.clip((x_cut - label_x) / 0.7, 0, 1)

        ann.set_alpha(fade)

        if ann.arrow_patch is not None:
            ann.arrow_patch.set_alpha(fade)



# =========================
# RENDER FRAMES
# =========================

frames = []

for i in range(N_FRAMES):
    update_frame(i)
    frames.append(fig_to_array(fig, alpha=ALPHA))

    if i % FPS == 0 or i == N_FRAMES - 1:
        print(f"[FRAME] {i + 1}/{N_FRAMES}")


# =========================
# EXPORT VIA VIZLIB
# =========================

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="m_dwarf_spectrum_reveal",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    cleanup_frames=True,
    webm_crf=28,
    mp4_crf=20,
    alpha=ALPHA,
)

plt.close(fig)

print(f"[OUT_FILE] {out_file}")
print(f"[EXISTS] {out_file.exists()}")
print(f"[SIZE] {out_file.stat().st_size if out_file.exists() else 'missing'}")

[FRAME] 1/300
[FRAME] 31/300
[FRAME] 61/300
[FRAME] 91/300
[FRAME] 121/300
[FRAME] 151/300
[FRAME] 181/300
[FRAME] 211/300
[FRAME] 241/300
[FRAME] 271/300
[FRAME] 300/300


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

[OUT_FILE] animations/dwarf_spectra/m_dwarf_spectrum_reveal.webm
[EXISTS] True
[SIZE] 228621


[out#0/webm @ 0x131e0ea10] video:221KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 1.151678%
frame=  300 fps= 59 q=28.0 Lsize=     223KiB time=00:00:10.00 bitrate= 182.9kbits/s speed=1.98x    
